<a href="https://colab.research.google.com/github/ravzanurcuhaci/medical-triage-system/blob/main/regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q deep-translator sentence-transformers

In [ ]:
import joblib
import numpy as np
import pandas as pd

model_dir = "/content/drive/MyDrive/medical_triage_project/models/"

baseline_model = joblib.load(model_dir + "classifier.pkl")
retrieval_embeddings = np.load(model_dir + "retrieval_embeddings.npy")
retrieval_df = pd.read_csv(model_dir + "retrieval_data.csv")

print("✅ Model ve veriler yüklendi")

In [ ]:
!pip install -q datasets huggingface_hub pandas

In [ ]:
from datasets import load_dataset

ds = load_dataset("gretelai/symptom_to_diagnosis")
ds

In [ ]:
train_df = ds["train"].to_pandas()
test_df = ds["test"].to_pandas()

print(train_df.head())
print(test_df.head())
print(train_df.columns)

In [ ]:
import os

save_path = "/content/drive/MyDrive/medical_triage_project/data/raw/"

os.makedirs(save_path, exist_ok=True)

In [ ]:

train_df.to_csv(save_path + "gretelai_train.csv", index=False)
test_df.to_csv(save_path + "gretelai_test.csv", index=False)

print("Kaydedildi.")

In [ ]:
import  pandas as pd
df = pd.read_csv("/content/drive/MyDrive/medical_triage_project/data/raw/gretelai_train.csv")

print(df.head())
print(df.columns)
print(df.shape)

In [ ]:
df1= pd.read_csv("Final_Augmented_dataset_Diseases_and_Symptoms.csv")

In [ ]:
df2= pd.read_csv("Symptom2Disease.csv")

In [ ]:
print(df1.head())

In [ ]:
print(df2.head())

In [ ]:
symptom2_std = pd.DataFrame({
    "text": df2["text"].astype(str).str.strip(),
    "label": df2["label"].astype(str).str.strip().str.lower(),
    "source": "symptom2disease",
    "style": "natural",
    "language": "en"
})

In [ ]:
gretelai_std = pd.DataFrame({
    "text": train_df["input_text"].astype(str).str.strip(),
    "label": train_df["output_text"].astype(str).str.strip().str.lower(),
    "source": "gretelai",
    "style": "natural",
    "language": "en"
})

In [ ]:
classifier_df = pd.concat([symptom2_std, gretelai_std], ignore_index=True)

In [ ]:
symptom_cols = [col for col in df1.columns if col != "diseases"]

print(len(symptom_cols))  # ~377 olması lazım

In [ ]:
def row_to_text(row, symptom_cols):
    active = []
    for col in symptom_cols:
        if row[col] == 1 or row[col] == 1.0:
            clean = col.strip().lower()
            active.append(clean)
    return "; ".join(active)

In [ ]:
structured_std = pd.DataFrame({
    "text": df1.apply(lambda row: row_to_text(row, symptom_cols), axis=1),
    "label": df1["diseases"].astype(str).str.strip().str.lower(),
    "source": "structured_dataset",
    "style": "structured",
    "language": "en"
})

In [ ]:
print(structured_std.head())
print(structured_std.shape)

In [ ]:
structured_std = structured_std[structured_std["text"] != ""]
structured_std = structured_std.reset_index(drop=True)

In [ ]:
save_path = "/content/drive/MyDrive/medical_triage_project/data/processed/"
import os
os.makedirs(save_path, exist_ok=True)

structured_std.to_csv(save_path + "retrieval_dataset.csv", index=False)

print("structured_std kaydedildi")

In [ ]:
print(classifier_df.head())
print(classifier_df.shape)
print(classifier_df["source"].value_counts())
print(classifier_df["label"].nunique())
print(classifier_df.isnull().sum())

In [ ]:
print(structured_std.head())
print(structured_std.shape)
print(structured_std["label"].nunique())
print(structured_std.isnull().sum())

In [ ]:
classifier_df = classifier_df.drop_duplicates().reset_index(drop=True)
structured_std = structured_std.drop_duplicates().reset_index(drop=True)

In [ ]:
print("classifier:", classifier_df.shape)
print("retrieval:", structured_std.shape)

In [ ]:
classifier_df["label"] = classifier_df["label"].astype(str).str.strip().str.lower()
structured_std["label"] = structured_std["label"].astype(str).str.strip().str.lower()

classifier_df["text"] = classifier_df["text"].astype(str).str.strip()
structured_std["text"] = structured_std["text"].astype(str).str.strip()

In [ ]:
classifier_labels = set(classifier_df["label"].unique())
retrieval_labels = set(structured_std["label"].unique())

print("Classifier label sayısı:", len(classifier_labels))
print("Retrieval label sayısı:", len(retrieval_labels))
print("Ortak label sayısı:", len(classifier_labels.intersection(retrieval_labels)))

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    classifier_df,
    test_size=0.2,
    stratify=classifier_df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["label"],
    random_state=42
)

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

In [ ]:
save_path = "/content/drive/MyDrive/medical_triage_project/data/processed/"

train_df.to_csv(save_path + "classifier_train.csv", index=False)
val_df.to_csv(save_path + "classifier_val.csv", index=False)
test_df.to_csv(save_path + "classifier_test.csv", index=False)

print("classifier splitleri kaydedildi")

In [ ]:
#split soyalarını oku

In [ ]:
import pandas as pd

save_path = "/content/drive/MyDrive/medical_triage_project/data/processed/"

train_df = pd.read_csv(save_path + "classifier_train.csv")
val_df   = pd.read_csv(save_path + "classifier_val.csv")
test_df  = pd.read_csv(save_path + "classifier_test.csv")

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

display(train_df.head())

In [ ]:
#x ve y ayır

In [ ]:
X_train = train_df["text"].astype(str)
y_train = train_df["label"].astype(str)

X_val = val_df["text"].astype(str)
y_val = val_df["label"].astype(str)

X_test = test_df["text"].astype(str)
y_test = test_df["label"].astype(str)

In [ ]:
#baseline kur
#Bu bir text classifier. Metni önce TF-IDF ile vektöre çeviriyor, sonra sınıflandırma yapıyor.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

baseline_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        max_features=20000
    )),
    ("clf", LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ))
])

baseline_model.fit(X_train, y_train)
print("Model eğitildi.")

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

val_pred = baseline_model.predict(X_val)

print("Validation Accuracy:", accuracy_score(y_val, val_pred))
print(classification_report(y_val, val_pred, zero_division=0))

In [ ]:
test_pred = baseline_model.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, test_pred))
print(classification_report(y_test, test_pred, zero_division=0))

In [ ]:
import numpy as np

classes = baseline_model.named_steps["clf"].classes_
probs = baseline_model.predict_proba(X_test)

def get_top_k_predictions(prob_row, classes, k=3):
    idx = np.argsort(prob_row)[-k:][::-1]
    return [(classes[i], float(prob_row[i])) for i in idx]

for i in range(5):
    print("TEXT:", X_test.iloc[i])
    print("TRUE LABEL:", y_test.iloc[i])
    print("TOP-3:", get_top_k_predictions(probs[i], classes, k=3))
    print("-" * 100)

In [ ]:
import os
import joblib

model_dir = "/content/drive/MyDrive/medical_triage_project/models/"
os.makedirs(model_dir, exist_ok=True)

joblib.dump(baseline_model, model_dir + "baseline_tfidf_logreg.pkl")
print("Model kaydedildi.")

In [ ]:
sample_text = "i feel like vomiting and stomach hurts"

pred = baseline_model.predict([sample_text])[0]
pred_probs = baseline_model.predict_proba([sample_text])[0]

print("Prediction:", pred)
print("Top-3:", get_top_k_predictions(pred_probs, classes, k=3))

In [ ]:
!pip install -q sentence-transformers

In [ ]:
import pandas as pd

processed_path = "/content/drive/MyDrive/medical_triage_project/data/processed/"

retrieval_df = pd.read_csv(processed_path + "retrieval_dataset.csv")

print(retrieval_df.shape)
display(retrieval_df.head())

In [ ]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding modeli yüklendi.")

In [ ]:
retrieval_texts = retrieval_df["text"].astype(str).tolist()

retrieval_embeddings = embed_model.encode(
    retrieval_texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

print(retrieval_embeddings.shape)

In [ ]:
import numpy as np
import os

model_dir = "/content/drive/MyDrive/medical_triage_project/models/"
os.makedirs(model_dir, exist_ok=True)

np.save(model_dir + "retrieval_embeddings.npy", retrieval_embeddings)
print("Retrieval embeddingleri kaydedildi.")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def retrieve_similar_cases(query_text, retrieval_df, retrieval_embeddings, embed_model, top_k=5):
    query_embedding = embed_model.encode([query_text], convert_to_numpy=True)

    sims = cosine_similarity(query_embedding, retrieval_embeddings)[0]
    top_idx = np.argsort(sims)[-top_k:][::-1]

    results = retrieval_df.iloc[top_idx].copy()
    results["similarity"] = sims[top_idx]

    return results[["text", "label", "similarity"]].reset_index(drop=True)

In [ ]:
query = "I have had cough, sore throat and fever for 3 days"

results = retrieve_similar_cases(
    query_text=query,
    retrieval_df=retrieval_df,
    retrieval_embeddings=retrieval_embeddings,
    embed_model=embed_model,
    top_k=5
)

display(results)

In [ ]:
import numpy as np

classes = baseline_model.named_steps["clf"].classes_

def get_top_k_predictions(prob_row, classes, k=3):
    idx = np.argsort(prob_row)[-k:][::-1]
    return [(classes[i], float(prob_row[i])) for i in idx]

def predict_with_retrieval(user_text, baseline_model, classes, retrieval_df, retrieval_embeddings, embed_model, top_k_cls=3, top_k_ret=5):
    pred_probs = baseline_model.predict_proba([user_text])[0]
    top_preds = get_top_k_predictions(pred_probs, classes, k=top_k_cls)

    retrieved = retrieve_similar_cases(
        query_text=user_text,
        retrieval_df=retrieval_df,
        retrieval_embeddings=retrieval_embeddings,
        embed_model=embed_model,
        top_k=top_k_ret
    )

    return {
        "input_text": user_text,
        "top_predictions": top_preds,
        "retrieved_cases": retrieved
    }

In [ ]:
user_text = "I have had cough, sore throat and fever for 3 days"

result = predict_with_retrieval(
    user_text=user_text,
    baseline_model=baseline_model,
    classes=classes,
    retrieval_df=retrieval_df,
    retrieval_embeddings=retrieval_embeddings,
    embed_model=embed_model,
    top_k_cls=3,
    top_k_ret=5
)

print("INPUT:")
print(result["input_text"])

print("\nTOP-3 PREDICTIONS:")
for label, score in result["top_predictions"]:
    print(f"{label}: {score:.4f}")

print("\nRETRIEVED CASES:")
display(result["retrieved_cases"])

In [ ]:
retrieved_labels = result["retrieved_cases"]["label"].value_counts()
print(retrieved_labels)

In [ ]:
# =========================
# FINAL INFERENCE BLOĞU
# classifier + retrieval + özet + final yorum
# =========================

import numpy as np
from collections import Counter

# 1) Top-k prediction fonksiyonu
def get_top_k_predictions(prob_row, classes, k=3):
    idx = np.argsort(prob_row)[-k:][::-1]
    return [(classes[i], float(prob_row[i])) for i in idx]

# 2) Retrieval fonksiyonu
def retrieve_similar_cases(query_text, retrieval_df, retrieval_embeddings, embed_model, top_k=5):
    query_embedding = embed_model.encode([query_text], convert_to_numpy=True)
    sims = cosine_similarity(query_embedding, retrieval_embeddings)[0]
    top_idx = np.argsort(sims)[-top_k:][::-1]

    results = retrieval_df.iloc[top_idx].copy()
    results["similarity"] = sims[top_idx]

    return results[["text", "label", "similarity"]].reset_index(drop=True)

# 3) Retrieval label özeti
def summarize_retrieved_labels(retrieved_cases):
    label_counts = retrieved_cases["label"].value_counts().to_dict()
    return label_counts

# 4) Final yorum üret
def build_final_comment(top_predictions, retrieved_label_summary):
    top1_label = top_predictions[0][0]
    top1_score = top_predictions[0][1]

    if len(retrieved_label_summary) == 0:
        return "No similar retrieved cases were found."

    retrieved_top_label = max(retrieved_label_summary, key=retrieved_label_summary.get)
    retrieved_top_count = retrieved_label_summary[retrieved_top_label]

    # classifier top-1 retrieval'da da destekleniyorsa
    if top1_label in retrieved_label_summary:
        return (
            f"Classifier top prediction is '{top1_label}' "
            f"(score={top1_score:.3f}) and retrieval also supports this label."
        )

    # classifier top-1 retrieval ile uyuşmuyorsa
    return (
        f"Classifier top prediction is '{top1_label}' (score={top1_score:.3f}), "
        f"but retrieved cases are more consistent with '{retrieved_top_label}' "
        f"({retrieved_top_count} similar case(s))."
    )

# 5) Ana fonksiyon
def full_predict(user_text, baseline_model, classes, retrieval_df, retrieval_embeddings, embed_model,
                 top_k_cls=3, top_k_ret=5):

    # classifier
    pred_probs = baseline_model.predict_proba([user_text])[0]
    top_predictions = get_top_k_predictions(pred_probs, classes, k=top_k_cls)

    # retrieval
    retrieved_cases = retrieve_similar_cases(
        query_text=user_text,
        retrieval_df=retrieval_df,
        retrieval_embeddings=retrieval_embeddings,
        embed_model=embed_model,
        top_k=top_k_ret
    )

    # retrieval özeti
    retrieved_label_summary = summarize_retrieved_labels(retrieved_cases)

    # basit semptom özeti
    summary = f"Patient-reported complaint: {user_text}"

    # final yorum
    final_comment = build_final_comment(top_predictions, retrieved_label_summary)

    return {
        "input_text": user_text,
        "summary": summary,
        "top_predictions": top_predictions,
        "retrieved_cases": retrieved_cases,
        "retrieved_label_summary": retrieved_label_summary,
        "final_comment": final_comment
    }

# 6) Sonucu güzel bastır
def print_full_result(result):
    print("INPUT:")
    print(result["input_text"])

    print("\nSUMMARY:")
    print(result["summary"])

    print("\nTOP PREDICTIONS:")
    for i, (label, score) in enumerate(result["top_predictions"], start=1):
        print(f"{i}. {label} -> {score:.4f}")

    print("\nRETRIEVED LABEL SUMMARY:")
    for label, count in result["retrieved_label_summary"].items():
        print(f"- {label}: {count}")

    print("\nFINAL COMMENT:")
    print(result["final_comment"])

    print("\nRETRIEVED CASES:")
    display(result["retrieved_cases"])

In [ ]:
user_text = "I have had cough, sore throat and fever for 3 days"

result = full_predict(
    user_text=user_text,
    baseline_model=baseline_model,
    classes=classes,
    retrieval_df=retrieval_df,
    retrieval_embeddings=retrieval_embeddings,
    embed_model=embed_model,
    top_k_cls=3,
    top_k_ret=5
)

print_full_result(result)

In [ ]:
!pip install deep-translator
from deep_translator import GoogleTranslator

def translate_to_en(text):
    return GoogleTranslator(source='auto', target='en').translate(text)

In [ ]:
test_text_tr = "3 gündür öksürüyorum, boğazım ağrıyor ve ateşim var"
translated = translate_to_en(test_text_tr)

print("ORIGINAL:", test_text_tr)
print("TRANSLATED:", translated)

In [ ]:
test_text_en = "I have had cough, sore throat and fever for 3 days"
translated_en = translate_to_en(test_text_en)

print("ORIGINAL:", test_text_en)
print("TRANSLATED:", translated_en)

In [ ]:
from deep_translator import GoogleTranslator

def translate_to_tr(text):
    try:
        return GoogleTranslator(source='auto', target='tr').translate(text)
    except Exception as e:
        print("Translation error:", e)
        return text

In [ ]:
def translate_top_predictions(top_predictions):
    translated = []
    for label, score in top_predictions:
        label_tr = translate_to_tr(label)
        translated.append((label_tr, score))
    return translated

In [ ]:
def translate_retrieved_cases(retrieved_cases):
    translated_df = retrieved_cases.copy()

    # label çevir
    translated_df["label_tr"] = translated_df["label"].apply(translate_to_tr)

    # text çevirisi istersen bunu da aç
    translated_df["text_tr"] = translated_df["text"].apply(translate_to_tr)

    return translated_df

In [ ]:
def localize_result_to_tr(result):
    localized = {}

    localized["input_text_original"] = result["input_text"]
    localized["input_text_tr"] = translate_to_tr(result["input_text"])
    localized["summary_tr"] = translate_to_tr(result["summary"])

    localized["top_predictions_tr"] = translate_top_predictions(result["top_predictions"])

    localized["pattern_note_tr"] = translate_to_tr(result["pattern_note"])
    localized["final_comment_tr"] = translate_to_tr(result["final_comment"])

    localized["retrieved_cases_tr"] = translate_retrieved_cases(result["retrieved_cases"])

    return localized

In [ ]:
def print_localized_result_tr(localized_result):
    print("KULLANICI GİRİŞİ:")
    print(localized_result["input_text_tr"])

    print("\nÖZET:")
    print(localized_result["summary_tr"])

    print("\nOLASI DURUMLAR:")
    for i, (label_tr, score) in enumerate(localized_result["top_predictions_tr"], start=1):
        print(f"{i}. {label_tr} -> {score:.4f}")

    print("\nÖRÜNTÜ NOTU:")
    print(localized_result["pattern_note_tr"])

    print("\nSON YORUM:")
    print(localized_result["final_comment_tr"])

    print("\nUYARI:")
    print("Bu çıktı yalnızca semptom temelli karar destek amaçlıdır. Tıbbi tanı değildir.")

    print("\nBENZER VAKALAR (TR):")
    display(localized_result["retrieved_cases_tr"][["text_tr", "label_tr", "similarity"]])

In [ ]:
def run_full_system_tr(user_text):
    # 1) kullanıcı girişini İngilizceye çevir
    translated_input = translate_to_en(user_text)

    # 2) İngilizce model pipeline'ını çalıştır
    result_en = predict_with_retrieval(
        user_text=translated_input,
        baseline_model=baseline_model,
        classes=classes,
        retrieval_df=retrieval_df,
        retrieval_embeddings=retrieval_embeddings,
        embed_model=embed_model,
        top_k_cls=3,
        top_k_ret=5
    )

    # 3) predict_with_retrieval sadece bu alanları dönüyor olabilir
    # eksik alanları ekleyelim
    if "summary" not in result_en:
        result_en["summary"] = f"Patient-reported complaint: {translated_input}"

    if "pattern_note" not in result_en:
        result_en["pattern_note"] = "Retrieved cases suggest a symptom similarity pattern."

    if "final_comment" not in result_en:
        result_en["final_comment"] = (
            "These outputs represent possible conditions based on symptom similarity "
            "and are not a medical diagnosis."
        )

    # 4) retrieval label summary yoksa üret
    if "retrieved_label_summary" not in result_en:
        result_en["retrieved_label_summary"] = result_en["retrieved_cases"]["label"].value_counts().to_dict()

    # 5) kullanıcı için Türkçeleştir
    result_tr = localize_result_to_tr(result_en)

    print("MODELİN GÖRDÜĞÜ İNGİLİZCE INPUT:")
    print(translated_input)

    print("\n--- KULLANICIYA GÖSTERİLEN TÜRKÇE ÇIKTI ---")
    print_localized_result_tr(result_tr)

    return result_en, result_tr

In [ ]:
result_en, result_tr = run_full_system_tr("3 gündür öksürüyorum, boğazım ağrıyor ve ateşim var")

In [ ]:
print([name for name in globals() if "predict" in name])

In [ ]:
def run_user_text_direct(user_text):
    # 1) kullanıcı metnini İngilizceye çevir
    translated_input = translate_to_en(user_text)

    # 2) modeli doğrudan bu metinle çalıştır
    result_en = predict_with_retrieval(
        user_text=translated_input,
        baseline_model=baseline_model,
        classes=classes,
        retrieval_df=retrieval_df,
        retrieval_embeddings=retrieval_embeddings,
        embed_model=embed_model,
        top_k_cls=3,
        top_k_ret=5
    )

    # 3) gösterim için alanları tamamla
    result_en["summary"] = translated_input
    result_en["pattern_note"] = "Retrieved cases suggest a symptom similarity pattern."
    result_en["final_comment"] = (
        "These outputs represent possible conditions based on the user's complaint "
        "and similarity matching. They are not a medical diagnosis."
    )

    if "retrieved_label_summary" not in result_en:
        result_en["retrieved_label_summary"] = result_en["retrieved_cases"]["label"].value_counts().to_dict()

    # 4) Türkçeleştir
    result_tr = localize_result_to_tr(result_en)

    # 5) çıktı ver
    print("KULLANICI HAM GİRİŞİ:")
    print(user_text)

    print("\nMODELİN GÖRDÜĞÜ İNGİLİZCE METİN:")
    print(translated_input)

    print("\n--- KULLANICIYA GÖSTERİLEN TÜRKÇE ÇIKTI ---")
    print_localized_result_tr(result_tr)

    return result_en, result_tr

In [ ]:
long_text = """
Dün arkadaşlarla pikniğe gittik. Akşam eve dönünce çok halsiz hissettim.
Gece boğazım yanmaya başladı. Sabah kalkınca öksürüğüm vardı, biraz ateşim çıkmış gibi hissettim.
Başım da ağrıyor. Çok ciddi mi bilmiyorum ama son iki gündür arttı gibi.
"""

result_en, result_tr = run_user_text_direct(long_text)

In [ ]:
result_en, result_tr = run_full_system_tr("Öksürüğüm, boğaz ağrım ve ateşim var")

In [ ]:
result_en, result_tr = run_full_system_tr("İdrar yaparken yanma ve ateşim var")

In [ ]:
result_en, result_tr = run_full_system_tr("Cildimde kaşıntı ve kızarıklık var")

In [ ]:
result_en, result_tr = run_full_system_tr("Mide bulantım ve karın ağrım var")

In [ ]:
result_en, result_tr = run_full_system_tr("Nefes darlığım ve göğsümde sıkışma var")

In [ ]:
import joblib
import numpy as np

model_dir = "/content/drive/MyDrive/medical_triage_project/models/"

joblib.dump(baseline_model, model_dir + "classifier.pkl")
np.save(model_dir + "retrieval_embeddings.npy", retrieval_embeddings)
retrieval_df.to_csv(model_dir + "retrieval_data.csv", index=False)

In [ ]:
# =========================
# FINAL DEMO
# =========================

user_input = input("Şikayetinizi yazın: ")

result_en, result_tr = run_full_system_tr(user_input)